In [ ]:
import sys
from pathlib import Path

import cooler

sys.path.insert(1, "..")
import pandas as pd

from config.eda import DataConfig
from RNADNA_background.features_preparation import (
    interactions_bining,
)

In [ ]:
# configuration of the dataset
data_conf_file = Path("../config/data_conf.ini")
params = DataConfig(data_conf_file)

# cool file to get bins coordinates
cool_file = params.hic_folder / Path(
    f"{params.cell_line}.mcool::resolutions/{params.bin_size}"
)
c = cooler.Cooler(str(cool_file))

## Binning background contacts

In [ ]:
# tables separated by DNA part chromosome
pc_contacts = []
for chromosome in params.chromosomes:
    file = params.data_path / Path(f"tables_by_dna/mrnas_chr{chromosome}.tsv")
    table = pd.read_csv(
        file,
        usecols=[
            "gene_name_un",
            "gene_type",
            "rna_chr",
            "dna_chr",
            "dna_start",
            "dna_end",
        ],
        sep="\t",
    )
    table = table[table["gene_type"] == "protein_coding"].reset_index(
        drop=True
    )
    pc_contacts.append(table)
pc_contacts = pd.concat(pc_contacts).reset_index(drop=True)
pc_contacts_trans = pc_contacts[
    pc_contacts["rna_chr"] != pc_contacts["dna_chr"]
]

pc_contacts_binned = interactions_bining(pc_contacts_trans, params)

In [ ]:
bins = c.bins()[:][["chrom", "start", "end"]]
bins = bins.rename({"chrom": "dna_chr"}, axis=1)
bins["bin"] = bins["start"] // params.bin_size
bins = bins[bins["dna_chr"] != "chrY"].reset_index(drop=True)
pc_contacts_binned = bins.merge(
    pc_contacts_binned, how="left", on=["dna_chr", "bin"]
).fillna(0)

In [ ]:
pc_contacts_binned.to_csv(
    params.data_path / f"binned_contacts_{params.bin_size}.tsv",
    sep="\t",
    index=False,
)